In [3]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

In [4]:
import pandas as pd

df_layoffs = pd.read_csv('../data/layoffs_events.csv')
df_news = pd.read_csv('../data/news_sentiment.csv')

print("Layoffs shape:", df_layoffs.shape)
print("\nLayoffs columns:", df_layoffs.columns.tolist())
print("\nLayoffs sample:")
print(df_layoffs.head(3))

print("\n---")
print("News shape:", df_news.shape)
print("\nNews columns:", df_news.columns.tolist())
print("\nNews sample:")
print(df_news.head(3))

Layoffs shape: (2470, 11)

Layoffs columns: ['company', 'location', 'layoff_count', 'date', 'pct_workforce', 'industry', 'source_url', 'stage', 'raised_mm', 'country', 'is_ai_company']

Layoffs sample:
        company     location  layoff_count        date pct_workforce  \
0   Panda Squad  SF Bay Area           6.0  2020-03-13           75%   
1  HopSkipDrive  Los Angeles           8.0  2020-03-13           10%   
2      Help.com       Austin          16.0  2020-03-16          100%   

       industry                                         source_url    stage  \
0      Consumer  https://twitter.com/danielsing er/status/12385...     Seed   
1  Transportat…  https://layoffs.fyi/2020/04/02/ hopskipdrive-l...  Unknown   
2       Support                                           LinkedIn     Seed   

  raised_mm        country  is_ai_company  
0    $1.00   United States          False  
1   $45.00   United States          False  
2    $6.00   United States          False  

---
News shape:

In [5]:
def create_layoff_chunk(row):
    company = row['company'] if pd.notna(row['company']) else 'Unknown company'
    location = row['location'] if pd.notna(row['location']) else 'Unknown location'
    count = f"{int(row['layoff_count'])} employees" if pd.notna(row['layoff_count']) else 'an unknown number of employees'
    date = row['date'] if pd.notna(row['date']) else 'unknown date'
    pct = f"({row['pct_workforce']} of workforce)" if pd.notna(row['pct_workforce']) else ''
    industry = row['industry'] if pd.notna(row['industry']) else 'Unknown'
    country = row['country'] if pd.notna(row['country']) else 'Unknown'
    ai = "an AI company" if row['is_ai_company'] == True else "a non-AI company"
    
    return f"{company}, based in {location} ({country}), laid off {count} {pct} on {date}. They operate in the {industry} industry and are {ai}."

df_layoffs['date'] = pd.to_datetime(df_layoffs['date']).dt.strftime('%B %d, %Y')
df_layoffs['chunk'] = df_layoffs.apply(create_layoff_chunk, axis=1)

print("Sample chunks:")
for chunk in df_layoffs['chunk'].head(3):
    print(chunk)
    print()

Sample chunks:
Panda Squad, based in SF Bay Area (United States), laid off 6 employees (75% of workforce) on March 13, 2020. They operate in the Consumer industry and are a non-AI company.

HopSkipDrive, based in Los Angeles (United States), laid off 8 employees (10% of workforce) on March 13, 2020. They operate in the Transportat… industry and are a non-AI company.

Help.com, based in Austin (United States), laid off 16 employees (100% of workforce) on March 16, 2020. They operate in the Support industry and are a non-AI company.



In [6]:
def create_news_chunk(row):
    date = row['date'] if pd.notna(row['date']) else 'unknown date'
    title = row['title'] if pd.notna(row['title']) else 'Unknown title'
    source = row['source'] if pd.notna(row['source']) else 'Unknown source'
    sentiment = row['sentiment_cat'] if pd.notna(row['sentiment_cat']) else 'unknown'
    description = row['description'] if pd.notna(row['description']) else ''
    
    chunk = f"On {date}, {source} published: '{title}'. The sentiment of this article was {sentiment}."
    if description:
        chunk += f" Description: {description}"
    
    return chunk

df_news['chunk'] = df_news.apply(create_news_chunk, axis=1)

print("Sample news chunks:")
for chunk in df_news['chunk'].head(3):
    print(chunk)
    print()

Sample news chunks:
On 2022-11-09, Reuters published: 'Twitter lays off roughly half its workforce after Elon Musk takeover'. The sentiment of this article was neutral.

On 2022-11-09, NYT published: 'Meta to cut more than 11,000 employees in biggest layoff in company history'. The sentiment of this article was negative.

On 2022-11-14, WSJ published: 'Amazon begins largest round of layoffs in company history'. The sentiment of this article was positive.



In [7]:
# Create summary chunks from pre-computed aggregations
import pandas as pd

df_layoffs_raw = pd.read_csv('../data/layoffs_events.csv')

# Top industries by total layoffs
industry_totals = df_layoffs_raw.groupby('industry')['layoff_count'].sum().sort_values(ascending=False).head(10)
industry_summary = "Summary of total layoffs by industry: " + ", ".join([f"{ind}: {int(count):,} employees" for ind, count in industry_totals.items()])

# Top companies by total layoffs  
company_totals = df_layoffs_raw.groupby('company')['layoff_count'].sum().sort_values(ascending=False).head(10)
company_summary = "Summary of top companies by total layoffs: " + ", ".join([f"{comp}: {int(count):,} employees" for comp, count in company_totals.items()])

# AI vs Non-AI totals
ai_totals = df_layoffs_raw.groupby('is_ai_company')['layoff_count'].sum()
ai_summary = f"AI vs Non-AI company layoffs: AI companies laid off {int(ai_totals.get(True, 0)):,} employees total, Non-AI companies laid off {int(ai_totals.get(False, 0)):,} employees total."

# Year totals
df_layoffs_raw['year'] = pd.to_datetime(df_layoffs_raw['date']).dt.year
year_totals = df_layoffs_raw.groupby('year')['layoff_count'].sum().sort_values(ascending=False)
year_summary = "Summary of total layoffs by year: " + ", ".join([f"{int(year)}: {int(count):,} employees" for year, count in year_totals.items()])

# Country totals
country_totals = df_layoffs_raw.groupby('country')['layoff_count'].sum().sort_values(ascending=False).head(5)
country_summary = "Summary of layoffs by country (top 5): " + ", ".join([f"{country}: {int(count):,} employees" for country, count in country_totals.items()])

summary_chunks = [industry_summary, company_summary, ai_summary, year_summary, country_summary]
summary_ids = [f"summary_{i}" for i in range(len(summary_chunks))]
summary_metadata = [{"source": "summary", "index": i} for i in range(len(summary_chunks))]

print("Summary chunks created:")
for chunk in summary_chunks:
    print(f"\n{chunk}")

Summary chunks created:

Summary of total layoffs by industry: Other: 86,943 employees, Retail: 79,301 employees, Hardware: 54,784 employees, Consumer: 49,811 employees, Transportat…: 42,157 employees, Food: 37,765 employees, Finance: 33,927 employees, Healthcare: 27,312 employees, Travel: 15,338 employees, Infrastructu…: 13,170 employees

Summary of top companies by total layoffs: Amazon: 49,624 employees, Oracle: 30,289 employees, Intel: 22,794 employees, Microsoft: 18,900 employees, Tesla: 14,500 employees, Google: 12,290 employees, Meta: 12,200 employees, SAP: 11,000 employees, Ericsson: 10,100 employees, Philips: 10,000 employees

AI vs Non-AI company layoffs: AI companies laid off 149,030 employees total, Non-AI companies laid off 392,263 employees total.

Summary of total layoffs by year: 2023: 170,324 employees, 2024: 85,104 employees, 2022: 82,083 employees, 2025: 78,255 employees, 2026: 67,527 employees, 2020: 51,808 employees, 2021: 6,192 employees

Summary of layoffs by cou

In [11]:
# Add sentiment summary
df_news_raw = pd.read_csv('../data/news_sentiment.csv')
sentiment_counts = df_news_raw['sentiment_cat'].value_counts()
avg_sentiment_2023 = df_news_raw[df_news_raw['date'].str.startswith('2023')]['sentiment'].mean()

sentiment_summary = f"Media sentiment analysis of tech layoff news: Out of {len(df_news_raw)} articles analyzed, {sentiment_counts.get('negative', 0)} were negative, {sentiment_counts.get('positive', 0)} were positive, and {sentiment_counts.get('neutral', 0)} were neutral. The average sentiment score during 2023 was {avg_sentiment_2023:.4f}, indicating predominantly negative coverage during the major layoff wave."

summary_chunks.append(sentiment_summary)
summary_ids.append("summary_5")
summary_metadata.append({"source": "summary", "index": 5})

print(sentiment_summary)

Media sentiment analysis of tech layoff news: Out of 306 articles analyzed, 150 were negative, 119 were positive, and 37 were neutral. The average sentiment score during 2023 was -0.0397, indicating predominantly negative coverage during the major layoff wave.


In [12]:
# Combine ALL chunks including summaries
layoff_chunks = df_layoffs['chunk'].tolist()
news_chunks = df_news['chunk'].tolist()

# Metadata for each chunk
layoff_metadata = [{"source": "layoffs", "index": i} for i in range(len(layoff_chunks))]
news_metadata = [{"source": "news", "index": i} for i in range(len(news_chunks))]

all_chunks = layoff_chunks + news_chunks + summary_chunks
all_metadata = layoff_metadata + news_metadata + summary_metadata
all_ids = [f"layoff_{i}" for i in range(len(layoff_chunks))] + \
          [f"news_{i}" for i in range(len(news_chunks))] + \
          summary_ids

print(f"Total layoff chunks: {len(layoff_chunks)}")
print(f"Total news chunks: {len(news_chunks)}")
print(f"Total summary chunks: {len(summary_chunks)}")
print(f"Total chunks: {len(all_chunks)}")

Total layoff chunks: 2470
Total news chunks: 306
Total summary chunks: 6
Total chunks: 2782


In [10]:
import subprocess
subprocess.run(['/Users/sashipraneethmuthyala/anaconda3/bin/python', 
                '-m', 'pip', 'install', 
                'chromadb', 'sentence-transformers', 'openai', 'python-dotenv'],
               capture_output=True)
print("Packages ready")

Packages ready


In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB with persistent storage
client = chromadb.PersistentClient(path="../chroma_db")

# Delete existing collection if it exists (clean rebuild)
try:
    client.delete_collection("layoffs_rag")
    print("Deleted existing collection")
except:
    pass

# Create fresh collection
collection = client.create_collection(
    name="layoffs_rag",
    metadata={"hnsw:space": "cosine"}
)

# Load embedding model
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed and store in batches of 100
batch_size = 100
total = len(all_chunks)

for i in range(0, total, batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids = all_ids[i:i+batch_size]
    batch_metadata = all_metadata[i:i+batch_size]
    
    embeddings = model.encode(batch_chunks).tolist()
    
    collection.add(
        documents=batch_chunks,
        embeddings=embeddings,
        ids=batch_ids,
        metadatas=batch_metadata
    )
    print(f"Processed {min(i+batch_size, total)}/{total} chunks")

print(f"\nVector store built successfully!")
print(f"Total documents in collection: {collection.count()}")

Deleted existing collection
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processed 100/2782 chunks
Processed 200/2782 chunks
Processed 300/2782 chunks
Processed 400/2782 chunks
Processed 500/2782 chunks
Processed 600/2782 chunks
Processed 700/2782 chunks
Processed 800/2782 chunks
Processed 900/2782 chunks
Processed 1000/2782 chunks
Processed 1100/2782 chunks
Processed 1200/2782 chunks
Processed 1300/2782 chunks
Processed 1400/2782 chunks
Processed 1500/2782 chunks
Processed 1600/2782 chunks
Processed 1700/2782 chunks
Processed 1800/2782 chunks
Processed 1900/2782 chunks
Processed 2000/2782 chunks
Processed 2100/2782 chunks
Processed 2200/2782 chunks
Processed 2300/2782 chunks
Processed 2400/2782 chunks
Processed 2500/2782 chunks
Processed 2600/2782 chunks
Processed 2700/2782 chunks
Processed 2782/2782 chunks

Vector store built successfully!
Total documents in collection: 2782


In [12]:
test_queries = [
    "Which companies had the most layoffs?",
    "What happened to tech layoffs in 2023?",
    "How did AI companies perform compared to non-AI companies?"
]

for query in test_queries:
    print(f"Query: {query}")
    results = collection.query(query_texts=[query], n_results=3)
    for doc in results['documents'][0]:
        print(f"  → {doc[:100]}...")
    print()

Query: Which companies had the most layoffs?
  → Laybuy, based in Auckland (New Zealand), laid off 45 employees  on July 28, 2022. They operate in th...
  → On 2022-11-14, WSJ published: 'Amazon begins largest round of layoffs in company history'. The senti...
  → Getaround, based in SF Bay Area (United States), laid off 100 employees (25% of workforce) on March ...

Query: What happened to tech layoffs in 2023?
  → On 2026-04-09, New York Post published: 'Bay Area-based tech company announces shocking layoff of ne...
  → On 2025-04-02, WSJ published: 'Tech layoffs accelerate in Q1 2025 amid tariff uncertainty and AI con...
  → On 2026-04-02, New York Post published: 'AI pushes 2026 tech layoffs past 50K in just three months, ...

Query: How did AI companies perform compared to non-AI companies?
  → On 2023-07-18, Bloomberg published: 'Google, Microsoft, Amazon racing to hire AI researchers despite...
  → On 2024-03-07, Reuters published: 'IBM to replace 7,800 jobs with AI over five ye